# Notebook 3: Agent Tools, DB State & `LLM-as-a-Judge` (**DeepEval**)

This notebook cleanly separates the three roles people often mix up:
1. **Part A — Agent Tools & DB State (`No LLM Judge needed!`):** Verify exact tool selection/ordering with DeepEval `ToolCorrectnessMetric` and confirm the backend `ORDERS_DB` state changed properly.
2. **Part B — `LLM-as-a-Judge` (`DeepEval GEval`) + Human Calibration:** Use an LLM Judge strictly for subjective criteria (polite support tone) and measure **`Human Agreement >= 85%`**.
3. **Part C — `pass@k` vs `pass^k`:** Simulate repeated trials to see why reliability compounds downward over multi-run tasks.

In [ ]:
import sys
import pandas as pd
import deepeval
from deepeval.metrics import GEval, ToolCorrectnessMetric
from deepeval.test_case import LLMTestCase, SingleTurnParams, ToolCall

if ".." not in sys.path:
    sys.path.append("..")

from lankashop import ORDERS_DB, ask_agent, get_deepeval_judge, reset_db, set_live_mode

USE_LIVE_API = False  # Set True for live OpenAI GEval judge
set_live_mode(USE_LIVE_API)
print("DeepEval version:", deepeval.__version__)

## Part A: Agent Tools & Backend DB State (`No LLM Judge!`)

In [ ]:
tool_metric = ToolCorrectnessMetric(threshold=1.0, should_consider_ordering=True, include_reason=False, async_mode=False)

agent_cases = [
    ("Valid Refund (ORD-100)", "Please refund my unopened laptop order ORD-100.", "none", ["get_order", "issue_refund"], "ORD-100", True),
    ("Denied Earbuds (ORD-200)", "Please refund my opened Wireless Earbuds order ORD-200.", "none", ["get_order"], "ORD-200", False),
    ("BUGGY AGENT (Skips get_order!)", "Please refund my opened Wireless Earbuds order ORD-200.", "bad_tool", ["get_order"], "ORD-200", False),
]

tool_rows = []
for name, query, bm, expected_tools, order_id, expected_refunded in agent_cases:
    reset_db()
    tr = ask_agent(query, bug_mode=bm)

    tc = LLMTestCase(
        input=query,
        actual_output=tr["answer"],
        tools_called=[ToolCall(name=t["name"], input_parameters=t["args"]) for t in tr["tools_called"]],
        expected_tools=[ToolCall(name=n) for n in expected_tools],
    )
    tool_metric.measure(tc)

    tool_rows.append({
        "case": name,
        "tools_called": [t["name"] for t in tr["tools_called"]],
        "tool_pass": tool_metric.is_successful(),
        "db_refunded": ORDERS_DB[order_id]["refunded"],
        "db_state_ok": ORDERS_DB[order_id]["refunded"] is expected_refunded,
    })

pd.DataFrame(tool_rows)

## Part B: `LLM-as-a-Judge` (`DeepEval GEval`) & Human Calibration
Notice the distinct roles:
- `query` ➔ **LankaShop Agent (System Under Test)** generates `answer`.
- **`GEval` (Examiner / Judge)** scores whether `answer` follows the polite-tone rubric (`PASS = 1` / `FAIL = 0`).
- **`human_pass` (Calibration Labels)** checks whether the Judge aligns with human reviewers (`>= 85%` target).

In [ ]:
geval = GEval(
    name="Polite Support",
    evaluation_params=[SingleTurnParams.INPUT, SingleTurnParams.ACTUAL_OUTPUT],
    evaluation_steps=["Check tone is polite and never insults the customer."],
    threshold=0.7,
    model=get_deepeval_judge(),
    async_mode=False,
)

labeled_samples = [
    ("Polite Refund", "Please refund my unopened laptop order ORD-100.", "none", 1),
    ("Polite Denial", "Please refund my opened Wireless Earbuds order ORD-200.", "none", 1),
    ("Rude Bug", "Please refund my opened Wireless Earbuds order ORD-200.", "rude_tone", 0),
]

judge_rows = []
for name, query, bm, human_pass in labeled_samples:
    reset_db()
    agent_answer = ask_agent(query, bug_mode=bm)["answer"]
    geval.measure(LLMTestCase(input=query, actual_output=agent_answer))
    judge_pass = int(geval.is_successful())

    judge_rows.append({
        "case": name,
        "agent_answer": agent_answer[:45] + "...",
        "judge_pass": judge_pass,
        "human_pass": human_pass,
        "agreed": judge_pass == human_pass,
    })

df_j = pd.DataFrame(judge_rows)
print(f"Judge-Human Agreement: {df_j['agreed'].mean():.1%} (Production Target >= 85%)")
df_j

## Part C: Agent Consistency — `pass@k` vs `pass^k`
If an agent succeeds with probability `p = 0.90` on a single run, let's simulate `k = 5` consecutive trials over 1,000 episodes:

In [ ]:
import random

random.seed(42)
p, k, episodes = 0.90, 5, 1000
at_least_one, all_five = 0, 0

for _ in range(episodes):
    trials = [random.random() < p for _ in range(k)]
    at_least_one += int(any(trials))
    all_five += int(all(trials))

print(f"Simulated pass@5 (at least 1 ok): {at_least_one / episodes:.1%}")
print(f"Simulated pass^5 (ALL 5 ok!):     {all_five / episodes:.1%}  <- Crucial for high-stakes refund/finance agents")